In [ ]:
## because ai bharat trained on lower scikit learn library so we have to downgrade

!pip uninstall -y scikit-learn
!pip install scikit-learn==1.3.2

In [ ]:
!pip install ftfy regex indic-nlp-library unidecode -q

In [ ]:
!pip install Unidecode

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import regex as re
import unicodedata
from ftfy import fix_text

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA = PROJECT_ROOT / "data" / "raw" / "chandigarh_all.csv"
INTERIM_DATA = PROJECT_ROOT / "data" / "interim" / "cleaned_2.csv"
FINAL_DATA = PROJECT_ROOT / "outputs" / "final_df.csv"
RELIGION_MODEL_ROOT = PROJECT_ROOT / "third_party" / "its_all_in_the_name_light_repo"

In [ ]:
df = pd.read_csv(RAW_DATA)

In [ ]:
#Step 1
cols_to_clean = [
    "elector_name",
    "father_or_husband_name",
    "ac_name",
    "parl_constituency",
    "main_town",
    "police_station",
    "mandal",
    "district",
    "polling_station_name",
    "polling_station_address"
]

In [ ]:
def clean_unicode_text(text):

    if pd.isna(text):
        return np.nan

    # convert to string
    text = str(text)

    # fix mojibake / encoding corruption
    text = fix_text(text)

    # normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # remove zero-width junk except joiners
    text = re.sub(r'[\u200B-\u200D\uFEFF]', ' ', text)

    # remove OCR garbage symbols
    text = re.sub(r'[§©®™¥£€¢¬]', ' ', text)

    # normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    # strip
    text = text.strip()

    return text

In [ ]:
for col in cols_to_clean:
    df[col] = df[col].apply(clean_unicode_text)

In [ ]:
allowed_pattern = re.compile(
    r"^[\p{Devanagari}\p{Latin}\p{M}\s\.\-\,\'\/()।:&]+$"
)

In [ ]:
def detect_ocr_error(text):

    if pd.isna(text):
        return 0

    text = str(text).strip()

    if allowed_pattern.match(text):
        return 0
    else:
        return 1

In [ ]:
df["ocr_name_error"] = (
    df["elector_name"]
    .apply(detect_ocr_error)
)

df["ocr_pname_error"] = (
    df["father_or_husband_name"]
    .apply(detect_ocr_error)
)

In [ ]:
df["ocr_name_error"].value_counts()

In [ ]:
df["ocr_pname_error"].value_counts()

In [ ]:
df[df["ocr_name_error"] == 1][
    ["elector_name"]
].head(20)

In [ ]:
##Step 2

!pip install indic-transliteration -q

In [ ]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

In [ ]:
transliterate(
    "राम कुमार",
    sanscript.DEVANAGARI,
    sanscript.ITRANS
)

In [ ]:
def transliterate_text(text):

    if pd.isna(text):
        return np.nan

    text = str(text)

    try:
        return transliterate(
            text,
            sanscript.DEVANAGARI,
            sanscript.ITRANS
        )

    except:
        return text

In [ ]:
for col in cols_to_clean:

    new_col = col + "_latin"

    df[new_col] = df[col].apply(transliterate_text)

In [ ]:
df[
    [
        "elector_name",
        "elector_name_latin"
    ]
].sample(10)

In [ ]:
df[
    [
        "elector_name",
        "elector_name_latin",
        "father_or_husband_name",
        "father_or_husband_name_latin"
    ]
].sample(10)

In [ ]:
##Step 3

def canonicalize_name(text):

    if pd.isna(text):
        return np.nan

    text = str(text)

    # lowercase
    text = text.lower()

    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # normalize whitespace early
    text = re.sub(r"\s+", " ", text)

    # normalize repeated vowels
    text = re.sub(r"aa+", "a", text)
    text = re.sub(r"ii+", "i", text)
    text = re.sub(r"uu+", "u", text)
    text = re.sub(r"ee+", "e", text)
    text = re.sub(r"oo+", "o", text)

    # common personal-name normalization
    text = re.sub(r"kumara\b", "kumar", text)
    text = re.sub(r"prasada\b", "prasad", text)
    text = re.sub(r"rama\b", "ram", text)
    text = re.sub(r"krishnaa?\b", "krishna", text)
    text = re.sub(r"deva\b", "dev", text)
    text = re.sub(r"mauryaa?\b", "maurya", text)

    # chandigarh / place normalization
    text = re.sub(r"chandiga dha", "chandigarh", text)
    text = re.sub(r"dha\b", "dh", text)
    text = re.sub(r"gara\b", "garh", text)

    # locality / polling station normalization
    text = re.sub(r"saiktara|sektara|sek tara", "sector", text)

    text = re.sub(r"kaloni", "colony", text)

    text = re.sub(r"phesa", "phase", text)

    text = re.sub(r"smala phlaita", "small flats", text)

    text = re.sub(r"vikasa nagara", "vikas nagar", text)

    text = re.sub(r"manimajara", "manimajra", text)

    text = re.sub(r"dadu majara", "dadumajra", text)

    text = re.sub(r"khudtha|khudta", "khudda", text)

    text = re.sub(r"ambedakara", "ambedkar", text)

    text = re.sub(r"avasa", "awas", text)

    text = re.sub(r"yojana", "yojna", text)

    text = re.sub(r"borda", "board", text)

    text = re.sub(r"housimga", "housing", text)

    text = re.sub(r"gram", "village", text)

    text = re.sub(r"vesta", "west", text)

    text = re.sub(r"purva", "east", text)

    text = re.sub(
        r"indastriyala eriya phera",
        "industrial area phase",
        text
    )

    # institutional/address normalization
    text = re.sub(r"rajakiya", "rajkiya", text)

    text = re.sub(r"skula", "school", text)

    text = re.sub(r"madala", "model", text)

    text = re.sub(r"siniyara", "senior", text)

    text = re.sub(r"sakaimdari|sekamdari", "secondary", text)

    text = re.sub(r"praimari", "primary", text)

    text = re.sub(r"midala", "middle", text)

    text = re.sub(r"kamara na|ruma na", "room no", text)

    text = re.sub(r"bildimga|biladimga", "building", text)

    text = re.sub(r"amganava di|amganaba di", "anganwadi", text)

    text = re.sub(r"kmyuniti saimtara", "community center", text)

    text = re.sub(r"stapha ruma", "staff room", text)

    text = re.sub(r"narsari", "nursery", text)

    text = re.sub(r"haॉla|hala", "hall", text)

    text = re.sub(r"kaॉleja|kaaleja|kaleja", "college", text)

    text = re.sub(r"pablika", "public", text)

    text = re.sub(r"kompalaiksa", "complex", text)

    # consonant normalization
    text = re.sub(r"simh|simha|sinh", "singh", text)

    text = re.sub(r"khanh", "khan", text)

    # muhammad variants
    text = re.sub(
        r"\b(md|mhd|mohd|mohamed|mohammad|mohammed)\b",
        "muhammad",
        text
    )

    # honorific normalization
    text = re.sub(r"\bsri\b", "shri", text)

    text = re.sub(r"\bshrimat[i]?\b", "shrimati", text)

    # normalize whitespace again
    text = re.sub(r"\s+", " ", text)

    # strip
    text = text.strip()

    return text

In [ ]:
latin_cols = [col + "_latin" for col in cols_to_clean]

In [ ]:
for col in latin_cols:

    new_col = col + "_canon"

    df[new_col] = df[col].apply(canonicalize_name)

In [ ]:
def fix_polling_station_name(text):

    if pd.isna(text):
        return np.nan

    text = str(text)

    # OCR artifact cleanup
    text = re.sub(r"kaॉloni", "colony", text)

    text = re.sub(r"ndastriyala eriya", "industrial area", text)

    text = re.sub(r"indastriyala eriya", "industrial area", text)

    text = re.sub(r"darabara", "darbar", text)

    text = re.sub(r"hal lo majara", "hallo majra", text)

    text = re.sub(r"kajahe di", "kajheri", text)

    text = re.sub(r"badahe di", "badheri", text)

    text = re.sub(r"bahallna", "behlana", text)

    text = re.sub(r"kishanaga dh", "kishangarh", text)

    text = re.sub(r"vikasa nagarh", "vikas nagar", text)

    # transit house normalization
    text = re.sub(r"blaka", "block", text)

    text = re.sub(r"\bpi\b", "p", text)

    text = re.sub(r"tramjita", "transit", text)

    text = re.sub(r"hausa", "house", text)

    return text

In [ ]:
df["polling_station_name_latin_canon"] = (
    df["polling_station_name_latin_canon"]
    .apply(fix_polling_station_name)
)

In [ ]:
def fix_polling_station_address(text):

    if pd.isna(text):
        return np.nan

    text = str(text).lower()

    # --------------------------------------------------
    # existing OCR fixes
    # --------------------------------------------------

    text = re.sub(r"kaॉloni", "colony", text)

    text = re.sub(r"ndastriyala eriya", "industrial area", text)

    text = re.sub(r"indastriyala eriya", "industrial area", text)

    text = re.sub(r"darabara", "darbar", text)

    text = re.sub(r"hal lo majara", "hallo majra", text)

    text = re.sub(r"kajahe di", "kajheri", text)

    text = re.sub(r"badahe di", "badheri", text)

    text = re.sub(r"bahallna", "behlana", text)

    text = re.sub(r"kishanaga dh", "kishangarh", text)

    text = re.sub(r"vikasa nagarh", "vikas nagar", text)

    # --------------------------------------------------
    # SCHOOL / GOVERNMENT NORMALIZATION
    # --------------------------------------------------

    text = re.sub(r"\brajkiya\b", "government", text)

    text = re.sub(r"\bhai school\b", "high school", text)

    text = re.sub(r"\bmaॉdala\b", "model", text)

    text = re.sub(r"\bmaॉdal\b", "model", text)

    text = re.sub(r"\bsai school\b", "senior secondary school", text)

    text = re.sub(r"\bsenior secondary the tata hart\b", "senior secondary school", text)

    text = re.sub(r"\bmiddle school\b", "middle school", text)

    text = re.sub(r"\bprimary school\b", "primary school", text)

    # --------------------------------------------------
    # PLACE NORMALIZATION
    # --------------------------------------------------

    text = re.sub(r"dadamajara|dadmajara|da du majara", "dadumajra", text)

    text = re.sub(r"khuddatha alishera", "khudda alisher", text)

    text = re.sub(r"gamva", "village", text)

    text = re.sub(r"saramgapura", "sarangpur", text)

    text = re.sub(r"rayapura kalam", "raipur kalan", text)

    text = re.sub(r"rayapura khurda", "raipur khurd", text)

    text = re.sub(r"mauli jagarh", "mauli jagran", text)

    text = re.sub(r"mauli jagaram", "mauli jagran", text)

    text = re.sub(r"manimajra tauna", "manimajra town", text)

    text = re.sub(r"poketa na", "pocket no", text)

    text = re.sub(r"nyu indara colony", "new indira colony", text)

    text = re.sub(r"nyu irndara colony", "new indira colony", text)

    text = re.sub(r"ram darabara", "ram darbar", text)

    # --------------------------------------------------
    # COMMON OCR WORD FIXES
    # --------------------------------------------------

    text = re.sub(r"kamara|kameti ruh|hall ruma|homa saimsa ruma", "room", text)

    text = re.sub(r"nombara", "number", text)

    text = re.sub(r"najadika|nazadika|lazadika", "near", text)

    text = re.sub(r"pulisa steshana", "police station", text)

    text = re.sub(r"saimtara|semtara|saimtrala", "center", text)

    text = re.sub(r"komosita", "composite", text)

    text = re.sub(r"angamvari", "anganwadi", text)

    text = re.sub(r"krecha", "creche", text)

    text = re.sub(r"sportasa complex", "sports complex", text)

    text = re.sub(r"ethaletika klaba", "athletic club", text)

    text = re.sub(r"pamjaba ema ela e haॉstala", "punjab mla hostel", text)

    text = re.sub(r"kanavainta school", "convent school", text)

    text = re.sub(r"maumta karamala", "mount carmel", text)

    text = re.sub(r"amkura nursery", "ankur nursery", text)

    text = re.sub(r"dev samaja", "dev samaj", text)

    text = re.sub(r"bala niketana", "bal niketan", text)

    text = re.sub(r"jamja ghara", "janj ghar", text)

    text = re.sub(r"ilaiktrasiti aphisa", "electricity office", text)

    text = re.sub(r"markita kameti", "market committee", text)

    text = re.sub(r"timbara marketa", "timber market", text)

    text = re.sub(r"moti ram arya", "moti ram arya", text)

    text = re.sub(r"tribyuna", "tribune", text)

    text = re.sub(r"guru nanaka khallsa", "guru nanak khalsa", text)

    text = re.sub(r"kampalaiksa", "complex", text)

    text = re.sub(r"saba steshana", "sub station", text)

    text = re.sub(r"saupnsa school", "st anns school", text)

    text = re.sub(r"shivalika", "shivalik", text)

    text = re.sub(r"britisha samjaya", "british school", text)

    text = re.sub(r"semta josapha", "saint joseph", text)

    text = re.sub(r"baipatista", "baptist", text)

    text = re.sub(r"saimta stiphana", "saint stephen", text)

    text = re.sub(r"beniyana tri", "banyan tree", text)

    text = re.sub(r"reyana antararashtriya", "ryan international", text)

    text = re.sub(r"kommersa college evama bizanasa", "commerce college and business", text)

    # --------------------------------------------------
    # sector formatting
    # --------------------------------------------------

    text = re.sub(r"sector\s*(\d+)([a-z])", r"sector \1 \2", text)

    text = re.sub(r"38west", "38 west", text)

    # --------------------------------------------------
    # whitespace cleanup
    # --------------------------------------------------

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df["polling_station_address_latin_canon"] = (
    df["polling_station_address_latin_canon"]
    .apply(fix_polling_station_address)
)

In [ ]:
# explicit one-time OCR fixes before cleaning

df["elector_name_latin_canon"] = (
    df["elector_name_latin_canon"]
    .str.lower()
    .str.replace(
        r"\blavisha khan n[aA]\b",
        "lavish khanna",
        regex=True
    )
)

In [ ]:
import re

def fix_elector_name(text):

    if pd.isna(text):
        return np.nan

    text = str(text).lower().strip()

    # ----------------------------------------
    # basic spacing cleanup
    # ----------------------------------------

    text = re.sub(r"\s+", " ", text)

    # ----------------------------------------
    # OCR split-word fixes
    # ----------------------------------------

    split_word_fixes = {

        r"\bkhan\s+na\b": "khanna",
        r"\bchan\s+da\b": "chanda",
        r"\bcham\s+da\b": "chanda",
        r"\bsam\s+dhu\b": "sandhu",
        r"\bshar\s+ma\b": "sharma",
        r"\bthap\s+a\b": "thapa",
        r"\baro\s+ra\b": "arora",
        r"\bcha\s+bra\b": "chhabra"

    }

    for pattern, replacement in split_word_fixes.items():
        text = re.sub(pattern, replacement, text)

    # explicit OCR fix
    text = re.sub(
        r"\blavisha\s+khanna\b",
        "lavish khanna",
        text
    )

    # ----------------------------------------
    # OCR merged / broken syllables
    # ----------------------------------------

    text = re.sub(r"n\s+ni", "nni", text)

    text = re.sub(r"nan\s+da", "nanda", text)

    text = re.sub(r"san\s+tosha", "santosh", text)

    # ----------------------------------------
    # repeated consonants cleanup
    # ----------------------------------------

    text = re.sub(r"kk+", "k", text)
    text = re.sub(r"rr+", "r", text)
    text = re.sub(r"tt+", "t", text)
    text = re.sub(r"dd+", "d", text)
    text = re.sub(r"ll+", "l", text)
    text = re.sub(r"mm+", "m", text)

    # ----------------------------------------
    # OCR transliteration cleanup
    # ----------------------------------------

    replacements = {

        "mitala": "mittal",
        "pimtu": "pintu",
        "jaina": "jain",
        "sugamti": "suganti",
        "bhabavana": "bhagwan",
        "rajabhara": "rajbhar",
        "kamvalajita": "kanwaljit",
        "pamde": "pandey",
        "goyala": "goyal",
        "gaganajota": "gaganjot",
        "samgita": "sangeeta",
        "jasapimdara": "jaspinder",
        "kausara": "kausar",
        "jaham": "jahan",
        "talavara": "talwar",
        "chamda": "chanda",
        "kuladipa": "kuldeep",
        "amarina": "amreen",
        "chugha": "chugh",
        "mamju": "manju",
        "simgala": "singla",
        "sarapharaja": "sarfaraz",
        "thakura": "thakur",
        "amkusha": "ankush",
        "guracharana": "gurcharan",
        "suradh": "suradha",
        "ravata": "rawat",
        "raya": "rai",
        "thida": "thind",
        "dhara": "dhar",
        "guraprita": "gurpreet",
        "shina": "sheena",
        "maushmi": "mausmi",
        "riesa": "riyas",
        "vajiha": "wajiha",
        "giyana": "giyan",
        "pradipa": "pradip",
        "ishvera": "ishver",
        "balavimdara": "balvinder",
        "paramavira": "paramvir",
        "hukma": "hukam",
        "bhima": "bhim",
        "haradipa": "hardeep",
        "mukesha": "mukesh",
        "shiva": "shiv",
        "saroja": "saroj",
        "pavana": "pawan",
        'khan nA': 'khanna',
        'khan na': 'khanna',
        'chan da': 'chanda',
        "dipika": "deepika"

    }

    for wrong, correct in replacements.items():

        text = re.sub(
            rf"\b{re.escape(wrong)}\b",
            correct,
            text
        )

    # ----------------------------------------
    # surname normalization
    # ----------------------------------------

    surname_fixes = {

        "singha": "singh",
        "kumara": "kumar",
        "khana": "khan",
        "khannaa": "khanna",
        "chandaa": "chanda"

    }

    for wrong, correct in surname_fixes.items():

        text = re.sub(
            rf"\b{wrong}\b",
            correct,
            text
        )

    # ----------------------------------------
    # protect valid names from schwa deletion
    # ----------------------------------------

    protected = {

        # common first names
        "sita", "gita", "rita", "anita",
        "sunita", "kavita", "savita",
        "babita", "sarita", "uma",
        "rama", "devi", "kaur",
        "maya", "gayatri", "shivani",
        "sumitra", "dipali", "rati",
        "tara", "sima", "ava",
        "baby", "anju",

        # surnames
        "khanna",
        "thapa",
        "sharma",
        "sandhu",
        "chanda",
        "arora",
        "talwar",
        "mittal",
        "singh",
        "kumar",
        "goyal",
        "pandey",
        "thakur",
        "rawat",
        "thind",
        "dhar"

    }

    # ----------------------------------------
    # remove artificial schwa-ending "a"
    # ----------------------------------------

    words = []

    for w in text.split():

        if (
            w.endswith("a")
            and len(w) > 4
            and w not in protected
        ):
            w = w[:-1]

        words.append(w)

    text = " ".join(words)

    # ----------------------------------------
    # final cleanup
    # ----------------------------------------

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df["elector_name_latin_canon"] = (
    df["elector_name_latin_canon"]
    .apply(fix_elector_name)
)

In [ ]:
def fix_father_husband_name(text):

    if pd.isna(text):
        return np.nan

    text = str(text).lower().strip()

    # ----------------------------------------
    # spacing / OCR junk
    # ----------------------------------------

    text = re.sub(r"\s+", " ", text)

    text = re.sub(r"nan\s+da", "nanda", text)

    # ----------------------------------------
    # repeated consonants
    # ----------------------------------------

    text = re.sub(r"kk+", "k", text)
    text = re.sub(r"rr+", "r", text)
    text = re.sub(r"tt+", "t", text)
    text = re.sub(r"dd+", "d", text)
    text = re.sub(r"ll+", "l", text)
    text = re.sub(r"mm+", "m", text)

    # ----------------------------------------
    # OCR transliteration cleanup
    # ----------------------------------------

    replacements = {

        "krrishna": "krishna",
        "chamda": "chand",
        "tophika": "tofiq",
        "datta": "dutt",
        "sharanajita": "sharanjit",
        "garevala": "grewal",
        "chettari": "chettri",
        "jogindra": "joginder",
        "je ara": "jr",
        "patiyala": "patial",
        "dalajita": "daljit",
        "chamchall": "chanchal",
        "paramimdara": "parminder",
        "nachattara": "nachhattar",
        "kuladipa": "kuldeep",
        "lakhamira": "lakhmir",
        "mohada": "mohd",
        "kaushika": "kaushik",
        "paramara": "parmar",
        "satanama": "satnam",
        "mahajana": "mahajan",
        "bamsala": "bansal",
        "kulavanta": "kulwant",
        "ghavana": "dhawan",
        "samdhu": "sandhu",
        "jasabira": "jasbir",
        "phula": "phool",
        "ashvani": "ashwani",
        "kohali": "kohli",
        "karmabira": "karmbir",
        "kamshi": "kanshi",
        "suda": "sood",
        "krrishnalala": "krishnalal",
        "samtokha": "santokh",
        "kamalajita": "kamaljit",
        "garga": "garg",
        "taraloka": "trlok",
        "chirakuta": "chirkut",
        "ravata": "rawat",
        "jhageshvara": "jhageshwar",
        "samjiva": "sanjeev",
        "ranajita": "ranjit",
        "malakita": "malkit",
        "shukrulla": "shukrullah",
        "mobina": "mobeen",
        "ahamada": "ahmad",
        "ema ara": "mr",
        "namda": "nand"

    }

    for wrong, correct in replacements.items():
        text = re.sub(
            rf"\b{re.escape(wrong)}\b",
            correct,
            text
        )

    # ----------------------------------------
    # surname normalization
    # ----------------------------------------

    text = re.sub(r"\bsingha\b", "singh", text)

    text = re.sub(r"\bkumara\b", "kumar", text)

    text = re.sub(r"\bvarma\b", "verma", text)

    # ----------------------------------------
    # remove artificial schwa-ending "a"
    # ----------------------------------------

    protected = {

        # common names
        "sita", "gita", "rita", "anita",
        "sunita", "kavita", "savita",
        "babita", "sarita", "uma",
        "rama", "devi", "kaur",
        "maya", "gayatri", "shivani",
        "sumitra", "dipali", "rati",
        "tara", "sima", "ava",
        "baby", "anju",

        # surnames
        "khanna",
        "thapa",
        "sharma",
        "sandhu",
        "chanda",
        "arora",
        "talwar",
        "mittal",
        "singh",
        "kumar",
        "goyal",
        "pandey",
        "thakur",
        "rawat",
        "thind",
        "dhar",
        "verma",
        "kohli",
        "bansal",
        "mahajan"

    }

    words = []

    for w in text.split():

        if (
            w.endswith("a")
            and len(w) > 4
            and w not in protected
        ):
            w = w[:-1]

        words.append(w)

    text = " ".join(words)

    # ----------------------------------------
    # final cleanup
    # ----------------------------------------

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df["father_or_husband_name_latin_canon"] = (
    df["father_or_husband_name_latin_canon"]
    .apply(fix_father_husband_name)
)

In [ ]:
INTERIM_DATA.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(INTERIM_DATA, index=False)

In [ ]:
## Step 4

# input column for religion model
df["religion_input_name"] = (
    df["father_or_husband_name_latin_canon"]
    .fillna("")
    .astype(str)
    .str.strip()
)

In [ ]:
religion_cols = [
    "predicted_religion",
    "prob_hindu",
    "prob_muslim",
    "prob_sikh",
    "prob_christian",
    "prob_jain",
    "prob_buddhist"
]

for col in religion_cols:
    df[col] = np.nan

In [ ]:
religion_input = pd.DataFrame({
    "name": df["religion_input_name"]
})

In [ ]:
##saving data in ai bharat folder

(RELIGION_MODEL_ROOT / "data").mkdir(parents=True, exist_ok=True)
religion_input.to_csv(
    RELIGION_MODEL_ROOT / "data" / "sample_data.csv",
    index=False,
    encoding="utf-8"
)

In [ ]:
!python "{RELIGION_MODEL_ROOT / 'code' / 'run_py38.py'}"

In [ ]:
import pandas as pd

pred = pd.read_csv(
    RELIGION_MODEL_ROOT / "data" / "predictions" / "sample_data.csv"
)

pred.head()

In [ ]:
df["predicted_religion"] = pred["predicted_religion"]

df["prob_hindu"] = pred["Hindu"]
df["prob_muslim"] = pred["Muslim"]
df["prob_sikh"] = pred["Sikh"]
df["prob_christian"] = pred["Christian"]
df["prob_jain"] = pred["Jain"]
df["prob_buddhist"] = pred["Buddhist"]

In [ ]:
df[[
    "religion_input_name",
    "predicted_religion",
    "prob_hindu",
    "prob_muslim",
    "prob_sikh"
]].head(10)

In [ ]:
## Q1
freq = df["predicted_religion"].value_counts()

print(freq)

In [ ]:
(
    df["predicted_religion"]
   .value_counts(normalize=True) * 100
).round(2)

In [ ]:
df = df[
    df["predicted_religion"].isin(
        ["Hindu", "Muslim", "Sikh"]
    )
]

In [ ]:
##STep 5

df["elector_tokens"] = (
    df["elector_name_latin_canon"]
    .fillna("")
    .str.split()
)

df["parent_tokens"] = (
    df["father_or_husband_name_latin_canon"]
    .fillna("")
    .str.split()
)

In [ ]:
df[[
    "elector_name_latin_canon",
    "father_or_husband_name_latin_canon",
    "elector_tokens",
    "parent_tokens"
]].head(10)

In [ ]:
honorifics = {
    "md",
    "mohd",
    "mohammad",
    "muhammad",
    "mohammed",
    "sri",
    "shri",
    "shriman",
    "shrimati",
    "smt",
    "begum",
    "bibi",
    "sardar",
    "kumari",
    "kumar",
    "ji",
        "singh",
    "kaur",
    "devi",
    "lal",
    "chand",
    "bai",
    "ben",
    "das",
    "dass"
}

In [ ]:
def extract_personal_name(elector_tokens, parent_tokens):

    elector_tokens = [
        t for t in elector_tokens
        if t not in honorifics
    ]

    parent_set = set(parent_tokens)

    remaining = [
        t for t in elector_tokens
        if t not in parent_set
    ]

    if len(remaining) == 0:
        return np.nan

    return remaining[0]

In [ ]:
for i in range(10):
    print(
        df.loc[i, "elector_name_latin_canon"],
        " | ",
        df.loc[i, "father_or_husband_name_latin_canon"],
        " --> ",
        extract_personal_name(
            df.loc[i, "elector_tokens"],
            df.loc[i, "parent_tokens"]
        )
    )

In [ ]:
df["personal_name"] = df.apply(
    lambda row: extract_personal_name(
        row["elector_tokens"],
        row["parent_tokens"]
    ),
    axis=1
)

In [ ]:
df[[
    "elector_name_latin_canon",
    "father_or_husband_name_latin_canon",
    "personal_name"
]].sample(20, random_state=42)

In [ ]:
def extract_parent_personal_name(parent_tokens):

    parent_tokens = [
        t for t in parent_tokens
        if t not in honorifics
    ]

    if len(parent_tokens) == 0:
        return np.nan

    return parent_tokens[0]

In [ ]:
df["parent_personal_name"] = df["parent_tokens"].apply(
    extract_parent_personal_name
)

In [ ]:
##Step 6

df["birth_year"] = (
    pd.to_numeric(df["year"], errors="coerce")
    - pd.to_numeric(df["age"], errors="coerce")
)

In [ ]:
df["birth_year"].describe()

In [ ]:
df["married_female"] = np.where(
    (df["sex"].str.lower() == "female") &
    (df["relationship"].str.lower() == "husband"),
    1,
    0
)

In [ ]:
df["ac_no"] = (
    df["ac_name"]
    .astype(str)
    .str.extract(r"(\d+)")
)

In [ ]:
df["pollingstation_no"] = (
    df["polling_station_name"]
    .astype(str)
    .str.extract(r"(\d+)")
)

In [ ]:
df["religion_group"] = df["predicted_religion"].replace({
    "Hindu": "Hindu-Sikh",
    "Sikh": "Hindu-Sikh",
    "Muslim": "Muslim"
})

In [ ]:
rank_df = df[[
    "personal_name",
    "birth_year",
    "sex",
    "religion_group",
    "state"
]].copy()

In [ ]:
rank_df = rank_df[
    rank_df["personal_name"].notna()
]

rank_df = rank_df[
    rank_df["birth_year"].notna()
]

In [ ]:
name_counts = (
    rank_df
    .groupby([
        "state",
        "religion_group",
        "sex",
        "birth_year",
        "personal_name"
    ])
    .size()
    .reset_index(name="count")
)

In [ ]:
name_counts = name_counts.sort_values([
    "state",
    "religion_group",
    "sex",
    "birth_year"
])

In [ ]:
name_counts["historical_count"] = (
    name_counts
    .groupby([
        "state",
        "religion_group",
        "sex",
        "personal_name"
    ])["count"]
    .cumsum()
)

In [ ]:
name_counts["historical_count_prior"] = (
    name_counts["historical_count"]
    - name_counts["count"]
)

In [ ]:
name_counts["firstname_rank"] = (
    name_counts
    .groupby([
        "state",
        "religion_group",
        "sex",
        "birth_year"
    ])["historical_count"]
    .rank(
        ascending=False,
        method="dense"
    )
)

In [ ]:
rank_merge = name_counts[[
    "state",
    "religion_group",
    "sex",
    "birth_year",
    "personal_name",
    "firstname_rank"
]]

In [ ]:
df = df.merge(
    rank_merge,
    on=[
        "state",
        "religion_group",
        "sex",
        "birth_year",
        "personal_name"
    ],
    how="left"
)

In [ ]:
df[[
    "personal_name",
    "birth_year",
    "religion_group",
    "sex",
    "firstname_rank"
]].sample(20, random_state=42)

In [ ]:
#Q2:
total_married_females = df["married_female"].sum()

print(total_married_females)

In [ ]:
mf_polling = (
    df.groupby("pollingstation_no")["married_female"]
    .sum()
)

In [ ]:
mf_polling = (
    df.groupby("pollingstation_no")["married_female"]
    .sum()
)

In [ ]:
##Q2 final

mf_polling.describe()

In [ ]:
max_count = mf_polling.max()

top_polling = mf_polling[
    mf_polling == max_count
]

print(top_polling)

In [ ]:
earliest_year = int(df["birth_year"].min())
latest_year = int(df["birth_year"].max())

print(earliest_year, latest_year)

In [ ]:
##Q3:

earliest_count = (
    df[df["birth_year"] == earliest_year]
    .shape[0]
)

latest_count = (
    df[df["birth_year"] == latest_year]
    .shape[0]
)

print("Earliest:", earliest_year, earliest_count)
print("Latest:", latest_year, latest_count)

In [ ]:
##Step 7 start

rni_df = df[[
    "state",
    "birth_year",
    "religion_group",
    "personal_name"
]].copy()

In [ ]:
rni_df = rni_df[
    rni_df["personal_name"].notna()
]

rni_df = rni_df[
    rni_df["birth_year"].notna()
]

In [ ]:
rni_df = rni_df[
    rni_df["religion_group"].isin(
        ["Muslim", "Hindu-Sikh"]
    )
]

In [ ]:
name_religion_counts = (
    rni_df
    .groupby([
        "state",
        "birth_year",
        "religion_group",
        "personal_name"
    ])
    .size()
    .reset_index(name="name_count")
)

In [ ]:
religion_totals = (
    rni_df
    .groupby([
        "state",
        "birth_year",
        "religion_group"
    ])
    .size()
    .reset_index(name="religion_total")
)

In [ ]:
name_probs = name_religion_counts.merge(
    religion_totals,
    on=[
        "state",
        "birth_year",
        "religion_group"
    ],
    how="left"
)

In [ ]:
name_probs["prob_name_given_religion"] = (
    name_probs["name_count"]
    /
    name_probs["religion_total"]
)

In [ ]:
rni_pivot = (
    name_probs
    .pivot_table(
        index=[
            "state",
            "birth_year",
            "personal_name"
        ],
        columns="religion_group",
        values="prob_name_given_religion",
        fill_value=0
    )
    .reset_index()
)


In [ ]:
rni_pivot["RNI"] = (
    rni_pivot["Muslim"]
    /
    (
        rni_pivot["Muslim"]
        + rni_pivot["Hindu-Sikh"]
    )
)

In [ ]:
rni_pivot["RNI_category"] = np.select(
    [
        rni_pivot["RNI"] <= 0.3,
        (rni_pivot["RNI"] > 0.3) &
        (rni_pivot["RNI"] <= 0.7),
        rni_pivot["RNI"] > 0.7
    ],
    [
        "Hindu-Sikh Name",
        "Ambiguous Name",
        "Muslim Name"
    ],
    default="Unknown"
)

In [ ]:
rni_merge = rni_pivot[[
    "state",
    "birth_year",
    "personal_name",
    "RNI",
    "RNI_category"
]]

In [ ]:
df = df.merge(
    rni_merge,
    on=[
        "state",
        "birth_year",
        "personal_name"
    ],
    how="left"
)

In [ ]:
df[[
    "personal_name",
    "birth_year",
    "RNI",
    "RNI_category"
]].sample(20, random_state=42)

In [ ]:
rni_pivot["decade"] = (
    (rni_pivot["birth_year"] // 10) * 10
).astype(int)

In [ ]:
rni_pivot[[
    "birth_year",
    "decade"
]].sample(10, random_state=42)

In [ ]:
##Q4:
target_names = [
    "balbir",
    "kusum",
    "sangeeta",
    "azad",
    "ram"
]

target_decades = [1970, 1980, 1990]

q4_df = rni_pivot[
    (rni_pivot["personal_name"].isin(target_names)) &
    (rni_pivot["decade"].isin(target_decades))
].copy()

In [ ]:
q4_df[[
    "birth_year",
    "decade",
    "personal_name",
    "RNI",
    "RNI_category"
]].sort_values(
    ["personal_name", "birth_year"]
)

In [ ]:
selected_years = [1975, 1985, 1995]

final_q4 = q4_df[
    q4_df["birth_year"].isin(selected_years)
].copy()

In [ ]:
final_q4["RNI"] = final_q4["RNI"].round(4)

In [ ]:
##Q4 Final
final_q4[[
    "birth_year",
    "personal_name",
    "RNI",
    "RNI_category"
]].sort_values(
    ["birth_year", "personal_name"]
)

In [ ]:
##STep 8
df["spousal"] = np.where(
    (df["sex"].str.lower() == "female") &
    (df["relationship"].str.lower() == "husband"),
    df["parent_personal_name"],
    np.nan
)


In [ ]:
df[[
    "sex",
    "relationship",
    "parent_personal_name",
    "spousal"
]].sample(20, random_state=42)

In [ ]:
parent_rni_df = df[
    df["predicted_religion"].isin(
        ["Hindu", "Muslim", "Sikh"]
    )
].copy()

In [ ]:
parent_rni_df["religion_group"] = (
    parent_rni_df["predicted_religion"]
    .replace({
        "Hindu": "Hindu-Sikh",
        "Sikh": "Hindu-Sikh",
        "Muslim": "Muslim"
    })
)

In [ ]:
parent_rni_df = parent_rni_df[[
    "state",
    "birth_year",
    "sex",
    "religion_group",
    "parent_personal_name",
    "spousal"
]]

In [ ]:
parent_rni_df = parent_rni_df[
    parent_rni_df["parent_personal_name"].notna()
]

In [ ]:
parent_name_counts = (
    parent_rni_df
    .groupby([
        "state",
        "birth_year",
        "religion_group",
        "parent_personal_name"
    ])
    .size()
    .reset_index(name="name_count")
)

In [ ]:
parent_religion_totals = (
    parent_rni_df
    .groupby([
        "state",
        "birth_year",
        "religion_group"
    ])
    .size()
    .reset_index(name="religion_total")
)

In [ ]:
parent_name_probs = parent_name_counts.merge(
    parent_religion_totals,
    on=[
        "state",
        "birth_year",
        "religion_group"
    ],
    how="left"
)

In [ ]:
parent_name_probs["prob_name_given_religion"] = (
    parent_name_probs["name_count"]
    /
    parent_name_probs["religion_total"]
)

In [ ]:
parent_rni_pivot = (
    parent_name_probs
    .pivot_table(
        index=[
            "state",
            "birth_year",
            "parent_personal_name"
        ],
        columns="religion_group",
        values="prob_name_given_religion",
        fill_value=0
    )
    .reset_index()
)

In [ ]:
parent_rni_pivot["parent_RNI"] = (
    parent_rni_pivot["Muslim"]
    /
    (
        parent_rni_pivot["Muslim"]
        +
        parent_rni_pivot["Hindu-Sikh"]
    )
)

In [ ]:
parent_rni_pivot["parent_RNI_category"] = np.select(
    [
        parent_rni_pivot["parent_RNI"] <= 0.3,
        (
            (parent_rni_pivot["parent_RNI"] > 0.3)
            &
            (parent_rni_pivot["parent_RNI"] <= 0.7)
        ),
        parent_rni_pivot["parent_RNI"] > 0.7
    ],
    [
        "Hindu-Sikh Name",
        "Ambiguous Name",
        "Muslim Name"
    ],
    default="Unknown"
)

In [ ]:
parent_rni_merge = parent_rni_pivot[[
    "state",
    "birth_year",
    "parent_personal_name",
    "parent_RNI",
    "parent_RNI_category"
]]

In [ ]:
df = df.merge(
    parent_rni_merge,
    on=[
        "state",
        "birth_year",
        "parent_personal_name"
    ],
    how="left"
)

In [ ]:
# Step 9

df["common_name_top3"] = np.where(
    df["firstname_rank"] <= 3,
    1,
    0
)

In [ ]:
df["common_name_top5"] = np.where(
    df["firstname_rank"] <= 5,
    1,
    0
)

In [ ]:
df["common_name_top10"] = np.where(
    df["firstname_rank"] <= 10,
    1,
    0
)

In [ ]:
df[[
    "personal_name",
    "firstname_rank",
    "common_name_top3",
    "common_name_top5",
    "common_name_top10"
]].sample(20, random_state=42)

In [ ]:
ac_name_counts = (
    df
    .groupby([
        "ac_name",
        "birth_year",
        "sex",
        "religion_group",
        "personal_name"
    ])
    .size()
    .reset_index(name="count")
)

In [ ]:
ac_name_counts["ac_rank"] = (
    ac_name_counts
    .groupby([
        "ac_name",
        "birth_year",
        "sex",
        "religion_group"
    ])["count"]
    .rank(
        method="dense",
        ascending=False
    )
)

In [ ]:
ac_name_counts["common_name_ac1000"] = np.where(
    ac_name_counts["ac_rank"] <= 1000,
    1,
    0
)

In [ ]:
ac_merge = ac_name_counts[[
    "ac_name",
    "birth_year",
    "sex",
    "religion_group",
    "personal_name",
    "common_name_ac1000"
]]

In [ ]:
df = df.merge(
    ac_merge,
    on=[
        "ac_name",
        "birth_year",
        "sex",
        "religion_group",
        "personal_name"
    ],
    how="left"
)

In [ ]:
## Q5 :

female_1985 = name_counts[
    (name_counts["state"] == "chandigarh") &
    (name_counts["birth_year"] == 1985) &
    (name_counts["sex"] == "Female") &
    (name_counts["religion_group"] == "Hindu-Sikh")
].sort_values("firstname_rank")

female_1985[
    [
        "personal_name",
        "firstname_rank"
    ]
].head(10)

In [ ]:
female_1985["personal_name"].head(3).tolist()

In [ ]:
female_1985["personal_name"].head(5).tolist()

In [ ]:
female_1985["personal_name"].head(10).tolist()

In [ ]:
## Step 10
df["is_muslim"] = np.where(
    df["religion_group"] == "Muslim",
    1,
    0
)

In [ ]:
df["parl_constituency_clean"] = "Chandigarh"

In [ ]:
pc_share = (
    df.groupby("parl_constituency_clean")["is_muslim"]
    .mean()
    .reset_index(name="pc_muslim_share")
)

pc_share.head()

In [ ]:
polling_share = (
    df.groupby("pollingstation_no")["is_muslim"]
    .mean()
    .reset_index(name="pollingstation_muslim_share")
)

polling_share.head()

In [ ]:
part_share = (
    df.groupby("part_no")["is_muslim"]
    .mean()
    .reset_index(name="part_muslim_share")
)

part_share.head()

In [ ]:
df = df.merge(
    polling_share,
    on="pollingstation_no",
    how="left"
)

In [ ]:
df = df.merge(
    pc_share,
    on="parl_constituency_clean",
    how="left"
)

In [ ]:
df = df.merge(
    part_share,
    on="part_no",
    how="left"
)

In [ ]:
df["segregation_pollingstation"] = (
    df["pollingstation_muslim_share"]
    - df["pc_muslim_share"]
)

In [ ]:
df["segregation_part"] = (
    df["part_muslim_share"]
    - df["pc_muslim_share"]
)

In [ ]:
df[
    [
        "pollingstation_no",
        "part_no",
        "pc_muslim_share",
        "pollingstation_muslim_share",
        "part_muslim_share",
        "segregation_pollingstation",
        "segregation_part"
    ]
].sample(20, random_state=42)

In [ ]:
## Q6 :

least_seg = df.loc[
    df["segregation_pollingstation"].abs().idxmin()
]

least_seg[
    [
        "pollingstation_no",
        "polling_station_name",
        "segregation_pollingstation",
        "pollingstation_muslim_share"
    ]
]

In [ ]:
max_part = df.loc[
    df["part_muslim_share"].idxmax()
]

max_part[
    [
        "part_no",
        "part_muslim_share"
    ]
]

In [ ]:
max_part[
    [
        "part_no",
        "pollingstation_no",
        "polling_station_name",
        "part_muslim_share"
    ]
]

In [ ]:
FINAL_DATA.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(FINAL_DATA, index=False)